# Friends' weekend — do we reply faster? (week 4, distributions)

Builds the Analysis 13 chart from `analysis-log.md`, for the week-4 distributions
assignment (`week4-distribution-requirements.md`).

**Question (Stage 1):** around the friends' weekend we have something to coordinate
("where are you?", "do you want a drink?") — do we reply to each other faster than on
an ordinary weekend?

**Primary statistic, fixed before looking:** per Fri–Sun block, the median reply gap;
the event weekend's median is at least 20% lower than ordinary Fri–Sun weekends.
Weekends with fewer than 2 reply gaps are left out.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 06-08: `timestamp` is stored as strings; the `+00:00` is only a label --
# these are the raw export's local clock times.
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)

NEW_CONVERSATION_MIN = 6 * 60  # a gap above this starts a new conversation, not a slow reply
MIN_REPLIES = 2  # a weekend needs at least this many reply gaps to count
FRIENDS_WEEKENDS = {"2024": pd.Timestamp("2024-04-19"), "2025": pd.Timestamp("2025-09-19")}

## Reply gap

A reply = the first message by a different author than the one before it (photos and
stickers count). The gap runs from the *last* message of the previous author's run.
Gaps over 6 hours start a new conversation and are not replies.

In [ ]:
minutes_since_previous = own["timestamp"].diff().dt.total_seconds() / 60
author_changed = own["author"].ne(own["author"].shift()) & minutes_since_previous.notna()

own["reply_gap_min"] = minutes_since_previous.where(author_changed)
own["new_conversation"] = own["reply_gap_min"] > NEW_CONVERSATION_MIN

replies = own[author_changed & ~own["new_conversation"]].copy()
# a Fri-Sun block is named after its Friday; a reply belongs to the day it was sent
replies = replies[replies["timestamp"].dt.weekday >= 4]
days_since_friday = pd.to_timedelta(replies["timestamp"].dt.weekday - 4, unit="D")
replies["weekend"] = replies["timestamp"].dt.normalize() - days_since_friday
replies["event"] = replies["weekend"].map({d: y for y, d in FRIENDS_WEEKENDS.items()}).fillna("none")

print(f"{len(replies)} Fri-Sun reply gaps over {replies['weekend'].nunique()} weekends")

## Stage 3 — shape (ordinary weekends only; event weekends not shown yet)

Two things to look at: the reply gaps themselves, and the per-weekend median that the
claim is about.

In [ ]:
ordinary = replies[replies["event"] == "none"]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(ordinary["reply_gap_min"], bins=range(0, NEW_CONVERSATION_MIN + 5, 5), color="grey")
ax.set_xlabel("reply gap (minutes, 5-minute bins)")
ax.set_ylabel("reply gaps")
ax.set_title("Reply gaps on ordinary Fri-Sun weekends")
plt.show()

print("share of reply gaps of exactly 0 minutes:", round((ordinary["reply_gap_min"] == 0).mean(), 2))
ordinary["reply_gap_min"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1)

In [ ]:
per_weekend = ordinary.groupby("weekend")["reply_gap_min"].agg(median="median", replies="size")
counted = per_weekend[per_weekend["replies"] >= MIN_REPLIES]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(counted["median"], bins=range(0, int(counted["median"].max()) + 2, 1), color="grey")
axes[0].set_xlabel("median reply gap of the weekend (minutes)")
axes[0].set_ylabel("weekends")
axes[0].set_title(f"{len(counted)} ordinary weekends with ≥ {MIN_REPLIES} reply gaps")
axes[1].hist(per_weekend["replies"], bins=30, color="grey")
axes[1].set_xlabel("reply gaps in the weekend")
axes[1].set_ylabel("weekends")
axes[1].set_title("How many reply gaps a weekend has")
plt.tight_layout()
plt.show()

print(f"weekends left out (< {MIN_REPLIES} reply gaps): {(per_weekend['replies'] < MIN_REPLIES).sum()}")
counted["median"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1)